In [ ]:
import numpy as np
import cmath

# ==========================================
# 1. SYSTEM CONSTANTS
# ==========================================
FS = 12500.0             
F_NOMINAL = 50.0         
V_NOMINAL_RMS = 220.0    
A_PEAK = V_NOMINAL_RMS * np.sqrt(2)

WINDOW_MS = 60.0
WINDOW_SAMPLES = int((WINDOW_MS / 1000.0) * FS)

# --- Test Points ---
TEST_CONFIG = {
    'VOLTAGE_FACTORS': [0.1, 1.2],
    'FREQ_POINTS':     [45.0, 50.0, 55.0], 
    'HARMONIC_ORDERS': [2, 3],
    'OOB_FACTORS':     [0.2, 1.95],
    'MOD_FREQS':       [2.0, 5.0],
    'RAMP_RANGE':      (45.0, 55.0),
    'STEP_MAGNITUDE':  0.1
}

# ==========================================
# 2. CORE ENGINE
# ==========================================
class PMUEngine:
    def __init__(self, reporting_fps=50):
        self.reporting_fps = reporting_fps
        self.dt_report = 1.0 / reporting_fps
        
        self.lut_start = 45.0
        self.lut_stop = 55.0
        self.lut_step = 0.2
        self.P_LUT = {}
        self.LUT_FREQS = []
        
        self.prev_freq_rocof = F_NOMINAL
        self._build_lut()

    def _generate_p_matrix(self, f_center):
        # t=0 is at index N/2
        t_win = np.linspace(-WINDOW_MS/2000.0, WINDOW_MS/2000.0, WINDOW_SAMPLES, endpoint=False)
        
        components = [
            (0, 1),           
            (f_center, 1),    
            (2*f_center, 0),  
            (3*f_center, 0)   
        ]
        
        H_cols = []
        col_names = []
        for freq, order in components:
            omega = 2 * np.pi * freq
            if freq == 0:
                H_cols.append(np.ones_like(t_win))
                col_names.append("DC_O0")
                if order >= 1:
                    H_cols.append(t_win)
                    col_names.append("DC_O1")
            else:
                H_cols.append(np.cos(omega * t_win))
                col_names.append("FUND_Cos_O0")
                H_cols.append(np.sin(omega * t_win))
                col_names.append("FUND_Sin_O0")
                if order >= 1:
                    H_cols.append(t_win * np.cos(omega * t_win))
                    col_names.append("FUND_Cos_O1")
                    H_cols.append(t_win * np.sin(omega * t_win))
                    col_names.append("FUND_Sin_O1")

        H = np.column_stack(H_cols)
        # Hanning Window
        w_vals = 0.5 * (1 - np.cos(2 * np.pi * np.arange(WINDOW_SAMPLES) / (WINDOW_SAMPLES - 1)))
        W = np.diag(w_vals)
        
        HT_W = H.T @ W
        HT_W_H = HT_W @ H
        HT_W_H += np.eye(HT_W_H.shape[0]) * 1e-12 
        
        P = np.linalg.inv(HT_W_H) @ HT_W
        return P, col_names

    def _build_lut(self):
        print(f"[{self.__class__.__name__}] Building LUT ({self.lut_start}-{self.lut_stop}Hz)...")
        steps = int(round((self.lut_stop - self.lut_start) / self.lut_step)) + 1
        freqs = np.linspace(self.lut_start, self.lut_stop, steps)
        for f in freqs:
            f_key = round(f, 2)
            P, names = self._generate_p_matrix(f_key)
            self.P_LUT[f_key] = (P, names)
            self.LUT_FREQS.append(f_key)

    def _get_nearest_matrix(self, f_est):
        f_clamped = max(self.lut_start, min(f_est, self.lut_stop))
        idx = int(round((f_clamped - self.lut_start) / self.lut_step))
        f_base = self.LUT_FREQS[idx]
        return self.P_LUT[f_base], f_base

    def _solve(self, chunk, P, col_names, f_base):
        x_hat = P @ chunk
        idx_c0 = col_names.index("FUND_Cos_O0")
        idx_s0 = col_names.index("FUND_Sin_O0")
        X_local = x_hat[idx_c0] - 1j * x_hat[idx_s0]
        
        idx_c1 = col_names.index("FUND_Cos_O1")
        idx_s1 = col_names.index("FUND_Sin_O1")
        X_deriv = x_hat[idx_c1] - 1j * x_hat[idx_s1]
        
        amp = np.abs(X_local)
        if amp > 0.1:
            f_dev = (1.0/(2*np.pi)) * (X_deriv * np.conj(X_local)).imag / (amp**2)
            f_est = f_base + f_dev
        else:
            f_est = f_base
        return X_local, f_est

    def process_frame(self, chunk, t_center_global):
        # 1. Coarse
        P_nom, names_nom = self.P_LUT[50.0]
        _, f_coarse = self._solve(chunk, P_nom, names_nom, 50.0)
        
        # 2. Fine
        (P_fine, names_fine), f_base_fine = self._get_nearest_matrix(f_coarse)
        X_local, f_final = self._solve(chunk, P_fine, names_fine, f_base_fine)
        
        # 3. Rotation Correction
        correction = np.exp(-1j * 2 * np.pi * F_NOMINAL * t_center_global)
        X_IEEE = X_local * correction
        
        # 4. ROCOF
        rocof = (f_final - self.prev_freq_rocof) / self.dt_report
        self.prev_freq_rocof = f_final
        
        v_rms = np.abs(X_IEEE) / np.sqrt(2)
        phase_deg = np.degrees(np.angle(X_IEEE))
        
        return v_rms, phase_deg, f_final, rocof

# ==========================================
# 3. ADC & BUFFER (FIXED TIMESTAMP)
# ==========================================
class ADCManager:
    def __init__(self, pmu_engine, buffer_size_sec=2.0):
        self.engine = pmu_engine
        self.buffer_len = int(buffer_size_sec * FS)
        self.ring_buffer = np.zeros(self.buffer_len)
        self.write_ptr = 0
        
        self.stride = int(FS / self.engine.reporting_fps)
        self.window_len = WINDOW_SAMPLES
        self.samples_since_last_report = 0
        
    def add_samples(self, new_data, t_start_global):
        reports = []
        n_new = len(new_data)
        
        for i, sample in enumerate(new_data):
            self.ring_buffer[self.write_ptr] = sample
            self.write_ptr = (self.write_ptr + 1) % self.buffer_len
            self.samples_since_last_report += 1
            
            if self.samples_since_last_report >= self.stride:
                self.samples_since_last_report = 0
                
                curr_ptr = self.write_ptr
                if curr_ptr >= self.window_len:
                    chunk = self.ring_buffer[curr_ptr - self.window_len : curr_ptr]
                else:
                    part1 = self.ring_buffer[curr_ptr - self.window_len :]
                    part2 = self.ring_buffer[:curr_ptr]
                    chunk = np.concatenate((part1, part2))
                
                # --- FIX: Precise Timestamp Calculation ---
                # t_current_sample is the time of the LAST sample (index N-1)
                t_current_sample = t_start_global + i/FS
                
                # With linspace(..., endpoint=False), the mathematical t=0 
                # is exactly at index N/2.
                # Delay = (N-1) - N/2 = N/2 - 1
                
                samples_delay = (WINDOW_SAMPLES / 2.0) - 1.0
                time_delay = samples_delay * (1.0 / FS)
                
                t_center = t_current_sample - time_delay
                
                result = self.engine.process_frame(chunk, t_center)
                reports.append((t_center, *result))
                
        return reports

# ==========================================
# 4. SIGNAL GENERATOR (Unchanged)
# ==========================================
class SignalGenerator:
    @staticmethod
    def steady_state(duration, volts, freq, harmonics=None):
        t = np.linspace(0, duration, int(duration*FS), endpoint=False)
        sig = (volts * np.sqrt(2)) * np.cos(2*np.pi*freq*t)
        
        if harmonics:
            phases = [-2*np.pi/3, 0, 2*np.pi/3] 
            for order, mag_ratio in harmonics:
                if isinstance(order, (int, np.integer)) and order % 1 == 0:
                    phase_idx = (int(order) - 2) % 3
                    h_phase = phases[phase_idx]
                else:
                    h_phase = 0.0
                h_mag = (volts * np.sqrt(2)) * mag_ratio
                sig += h_mag * np.cos(2*np.pi*(freq*order)*t + h_phase)
        return t, sig

    @staticmethod
    def modulation(duration, carrier_freq, am_params=None, pm_params=None):
        t = np.linspace(0, duration, int(duration*FS), endpoint=False)
        amp = A_PEAK
        phase = 0
        if am_params:
            k_a, f_m = am_params
            amp = A_PEAK * (1 + k_a * np.cos(2*np.pi*f_m*t))
        if pm_params:
            k_p, f_m = pm_params
            phase = k_p * np.cos(2*np.pi*f_m*t - np.pi) 
        sig = amp * np.cos(2*np.pi*carrier_freq*t + phase)
        return t, sig

    @staticmethod
    def ramp(start_f, end_f, ramp_rate):
        ramp_dur = abs(end_f - start_f) / ramp_rate
        hold_time = 0.5
        total_dur = hold_time + ramp_dur + hold_time
        t = np.linspace(0, total_dur, int(total_dur*FS), endpoint=False)
        
        inst_freq = np.zeros_like(t)
        phase_accum = np.zeros_like(t)
        
        idx_start = int(hold_time * FS)
        idx_end = int((hold_time + ramp_dur) * FS)
        
        inst_freq[:idx_start] = start_f
        phase_accum[:idx_start] = 2*np.pi * start_f * t[:idx_start]
        
        dt = 1/FS
        curr_phase = phase_accum[idx_start-1]
        
        for i in range(idx_start, len(t)):
            if i < idx_end:
                t_rel = t[i] - t[idx_start]
                if start_f < end_f: f_curr = start_f + ramp_rate * t_rel
                else:               f_curr = start_f - ramp_rate * t_rel
            else:
                f_curr = end_f
            inst_freq[i] = f_curr
            curr_phase += 2*np.pi * f_curr * dt
            phase_accum[i] = curr_phase
            
        sig = A_PEAK * np.cos(phase_accum)
        return t, sig, inst_freq, phase_accum

    @staticmethod
    def step(type, magnitude, phase_step):
        duration = 1.0
        t = np.linspace(0, duration, int(duration*FS), endpoint=False)
        idx_step = int(0.5 * FS)
        if type == 'mag':
            amps = np.ones_like(t) * A_PEAK
            amps[idx_step:] *= (1.0 + magnitude) 
            sig = amps * np.cos(2*np.pi*F_NOMINAL*t)
        elif type == 'phase':
            phases = np.zeros_like(t)
            phases[idx_step:] = np.radians(phase_step)
            sig = A_PEAK * np.cos(2*np.pi*F_NOMINAL*t + phases)
        return t, sig



In [2]:
# ==========================================
# 5. TEST SUITE
# ==========================================
class TestSuite:
    def __init__(self):
        self.engine = PMUEngine(reporting_fps=50)
        self.adc = ADCManager(self.engine)
        
    def run_test(self, test_name, t, signal, truth_func_complex=None, truth_func_f=None, truth_func_rocof=None, tolerance_tve=1.0, exclude_range=None):
        print(f"\n[Running] {test_name}")
        self.engine.prev_freq = F_NOMINAL
        self.engine.prev_freq_rocof = F_NOMINAL
        
        chunk_size = 1000 
        all_reports = []
        for i in range(0, len(signal), chunk_size):
            chunk_sig = signal[i : i+chunk_size]
            t_start = t[i]
            reps = self.adc.add_samples(chunk_sig, t_start)
            all_reports.extend(reps)
            
        max_tve = 0.0
        max_fe = 0.0
        max_rfe = 0.0
        
        # Header includes ROCOF and RFE
        print(f"{'Time':<8} | {'Vrms':<8} | {'Phase':<8} | {'Freq':<8} | {'ROCOF':<8} | {'TVE%':<8} | {'FE':<8} | {'RFE':<8}")
        print("-" * 95)
        
        skip = 5 
        
        for i, rep in enumerate(all_reports):
            if i < skip: continue
            r_time, r_vrms, r_phase, r_freq, r_rocof = rep
            
            # --- EXCLUSION INTERVAL CHECK ---
            # If r_freq is outside [min, max], we mark it excluded and do not update MAX errors.
            is_excluded = False
            if exclude_range:
                min_valid, max_valid = exclude_range
                if r_freq < min_valid or r_freq > max_valid:
                    is_excluded = True

            meas_vec = r_vrms * np.exp(1j * np.radians(r_phase))
            
            if truth_func_complex: true_vec = truth_func_complex(r_time)
            else: true_vec = V_NOMINAL_RMS
            
            if truth_func_f: true_freq = truth_func_f(r_time)
            else: true_freq = F_NOMINAL

            if truth_func_rocof: true_rocof = truth_func_rocof(r_time)
            else: true_rocof = 0.0
            
            # Robust TVE calculation
            if isinstance(true_vec, complex):
                diff = np.abs(meas_vec - true_vec)
                denom = np.abs(true_vec)
            else:
                diff = np.abs(meas_vec - true_vec) 
                denom = true_vec
                
            tve = (diff / (denom + 1e-9)) * 100.0
            fe = np.abs(r_freq - true_freq)
            rfe = np.abs(r_rocof - true_rocof)
            
            if not is_excluded:
                max_tve = max(max_tve, tve)
                max_fe = max(max_fe, fe)
                max_rfe = max(max_rfe, rfe)
            
            # Print periodically OR if excluded (to visualize transitions)
            if i % 20 == 0 or is_excluded: 
                excl_mark = "*" if is_excluded else " "
                print(f"{r_time:.3f}    | {r_vrms:.2f}   | {r_phase:.3f}    | {r_freq:.3f}    | {r_rocof:.3f}    | {tve:.3f}{excl_mark}   | {fe:.4f}   | {rfe:.4f}")
                
        print("-" * 95)
        status = "PASS" if max_tve < tolerance_tve else "FAIL"
        print(f"Result: {status} (Max TVE: {max_tve:.4f}%, Max FE: {max_fe:.4f} Hz, Max RFE: {max_rfe:.4f} Hz/s)")
        if exclude_range:
            print(f"(Exclusion Range applied: {exclude_range} Hz. '*' denotes excluded frames)")

    def execute_all(self):
        # 1. VOLTAGE SWEEP
        print("\n=== STEADY STATE: VOLTAGE SWEEP (10% - 120%) ===")
        for v_scale in [0.1, 0.8, 0.9, 1.1, 1.2]:
            t, sig = SignalGenerator.steady_state(0.5, V_NOMINAL_RMS*v_scale, F_NOMINAL)
            self.run_test(f"Voltage {v_scale*100}%", t, sig, 
                          truth_func_complex=lambda x: V_NOMINAL_RMS*v_scale, 
                          truth_func_rocof=lambda x: 0.0,
                          tolerance_tve=1.0)

        # 2. FREQUENCY SWEEP
        print("\n=== STEADY STATE: FREQUENCY SWEEP (45Hz - 55Hz) ===")
        for f in [45.0, 45.1, 49.95, 50.0, 50.05, 54.9, 55.0]:
            t, sig = SignalGenerator.steady_state(0.5, V_NOMINAL_RMS, f)
            tf = lambda tm: V_NOMINAL_RMS * np.exp(1j * 2*np.pi*(f-50.0)*tm)
            self.run_test(f"Freq {f}Hz", t, sig, 
                          truth_func_complex=tf, 
                          truth_func_f=lambda x: f, 
                          truth_func_rocof=lambda x: 0.0,
                          tolerance_tve=1.0)

        # 3. HARMONICS
        print("\n=== STEADY STATE: HARMONICS (10% Mag) ===")
        for h_order in [2, 3, 4, 50]:
            t, sig = SignalGenerator.steady_state(0.5, V_NOMINAL_RMS, F_NOMINAL, harmonics=[(h_order, 0.1)])
            self.run_test(f"Harmonic Order {h_order}", t, sig, 
                          truth_func_complex=lambda x: V_NOMINAL_RMS, 
                          truth_func_rocof=lambda x: 0.0,
                          tolerance_tve=1.0) 

        # 4. OUT OF BAND (OOB)
        print("\n=== STEADY STATE: OUT-OF-BAND (10-25Hz, 75-100Hz) ===")
        for h_order in [0.2, 0.45, 0.49, 1.5, 1.9]:
            t, sig_oob = SignalGenerator.steady_state(0.5, V_NOMINAL_RMS, F_NOMINAL, harmonics=[(h_order, 0.1)]) 
            self.run_test(f"OOB {h_order*50}Hz", t, sig_oob, 
                          truth_func_complex=lambda x: V_NOMINAL_RMS, 
                          truth_func_rocof=lambda x: 0.0,
                          tolerance_tve=1.3)
       
        # 5. AM MODULATION
        print("\n=== DYNAMIC: AM MODULATION (10% Depth, 2Hz) ===")
        t, sig_am = SignalGenerator.modulation(1.0, F_NOMINAL, am_params=(0.1, 2.0))
        tf_am = lambda tm: (V_NOMINAL_RMS * (1 + 0.1*np.cos(2*np.pi*2.0*tm)))
        self.run_test("AM Mod 2Hz", t, sig_am, 
                      truth_func_complex=tf_am, 
                      truth_func_rocof=lambda x: 0.0,
                      tolerance_tve=3.0)

        # 6. RAMP POSITIVE (45->55)
        # Exclusion: 45.04 to 54.96 Hz
        ramp_exclusion = (45.04, 54.96)
        
        print("\n=== DYNAMIC: RAMP POSITIVE (45->55Hz, 1Hz/s) ===")
        t, sig_ramp, f_profile, phase_profile = SignalGenerator.ramp(45.0, 55.0, 1.0)
        
        def get_ramp_phasor_pos(tm):
            idx = int(round(tm * FS))
            if idx >= len(phase_profile): idx = len(phase_profile) - 1
            abs_phase = phase_profile[idx]
            ref_phase = abs_phase - 2*np.pi*F_NOMINAL*tm
            return V_NOMINAL_RMS * np.exp(1j * ref_phase)

        def get_ramp_freq_pos(tm):
            if tm < 0.5: return 45.0
            elif tm < 10.5: return 45.0 + 1.0*(tm-0.5)
            else: return 55.0
            
        def get_ramp_rocof_pos(tm):
            if tm < 0.5: return 0.0
            elif tm < 10.5: return 1.0
            else: return 0.0

        self.run_test("Ramp Pos 45-55Hz", t, sig_ramp, 
                      truth_func_complex=get_ramp_phasor_pos, 
                      truth_func_f=get_ramp_freq_pos, 
                      truth_func_rocof=get_ramp_rocof_pos, 
                      tolerance_tve=1.0,
                      exclude_range=ramp_exclusion) 

        # 7. RAMP NEGATIVE (55->45)
        print("\n=== DYNAMIC: RAMP NEGATIVE (55->45Hz, 1Hz/s) ===")
        t, sig_ramp_neg, f_profile_neg, phase_profile_neg = SignalGenerator.ramp(55.0, 45.0, 1.0)
        
        def get_ramp_phasor_neg(tm):
            idx = int(round(tm * FS))
            if idx >= len(phase_profile_neg): idx = len(phase_profile_neg) - 1
            abs_phase = phase_profile_neg[idx]
            ref_phase = abs_phase - 2*np.pi*F_NOMINAL*tm
            return V_NOMINAL_RMS * np.exp(1j * ref_phase)

        def get_ramp_freq_neg(tm):
            if tm < 0.5: return 55.0
            elif tm < 10.5: return 55.0 - 1.0*(tm-0.5)
            else: return 45.0
            
        def get_ramp_rocof_neg(tm):
            if tm < 0.5: return 0.0
            elif tm < 10.5: return -1.0
            else: return 0.0

        self.run_test("Ramp Neg 55-45Hz", t, sig_ramp_neg, 
                      truth_func_complex=get_ramp_phasor_neg, 
                      truth_func_f=get_ramp_freq_neg, 
                      truth_func_rocof=get_ramp_rocof_neg, 
                      tolerance_tve=1.0,
                      exclude_range=ramp_exclusion) 

        # 8. STEP CHANGE
        print("\n=== DYNAMIC: MAGNITUDE STEP (+10%) ===")
        t, sig_step = SignalGenerator.step('mag', 0.1, 0)
        self.run_test("Step Mag +10%", t, sig_step, tolerance_tve=100.0) 

if __name__ == "__main__":
    suite = TestSuite()
    suite.execute_all()

[PMUEngine] Building LUT (45.0-55.0Hz)...

=== STEADY STATE: VOLTAGE SWEEP (10% - 120%) ===

[Running] Voltage 10.0%
Time     | Vrms     | Phase    | Freq     | ROCOF    | TVE%     | FE       | RFE     
-----------------------------------------------------------------------------------------------
0.390    | 22.00   | 0.000    | 50.000    | 0.000    | 0.000    | 0.0000   | 0.0000
-----------------------------------------------------------------------------------------------
Result: PASS (Max TVE: 0.0000%, Max FE: 0.0000 Hz, Max RFE: 0.0000 Hz/s)

[Running] Voltage 80.0%
Time     | Vrms     | Phase    | Freq     | ROCOF    | TVE%     | FE       | RFE     
-----------------------------------------------------------------------------------------------
0.390    | 176.00   | 0.000    | 50.000    | 0.000    | 0.000    | 0.0000   | 0.0000
-----------------------------------------------------------------------------------------------
Result: PASS (Max TVE: 0.0000%, Max FE: 0.0000 Hz, Max RFE: 

In [ ]:
# ==========================================
# 5. COMPLIANCE BENCH
# ==========================================
class ComplianceBench:
    def __init__(self):
        self.engine = PMUEngine()
        self.adc = ADCManager(self.engine)
        self.gen = SignalGenerator()
        
        # Limits
        self.FE_LIMIT = 0.01
        self.TVE_LIMIT = 1.0
        self.RESP_TIME_LIMIT = 0.040 # 40ms

    def run_case(self, title, test_type, params, duration=1.0):
        print(f"[Running] {title}")
        print(f"{'Time':<8} | {'Vrms':<8} | {'Phase':<8} | {'Freq':<8} | {'ROCOF':<8} | {'TVE%':<8} | {'FE':<8} | {'RFE':<8}")
        print("-" * 96)
        
        # Reset Engine State for clean start
        self.engine.prev_time = -1.0
        self.adc.write_ptr = 0
        self.adc.samples_accumulated = 0
        
        # We simulate in chunks (e.g., 20ms chunks)
        chunk_dur = 0.02
        t_current = 0.0
        
        stats = {'max_tve': 0.0, 'max_fe': 0.0, 'step_settled_time': None}
        
        while t_current < duration:
            # 1. Generate Raw Samples
            sig_chunk = self.gen.get_batch(t_current, chunk_dur, test_type, params)
            
            # 2. Feed ADC -> Get Reports
            reports = self.adc.add_samples(sig_chunk, t_current)
            
            # 3. Analyze Reports
            for rep in reports:
                t_rep, v_meas, ph_meas, f_meas, rocof_meas = rep
                
                # Get Reference at exact report time
                ref_mag, ref_ph, ref_freq, ref_rocof = self.gen.get_ref_at_t(t_rep, test_type, params)
                
                # Calc TVE
                ref_r = ref_mag * np.cos(ref_ph)
                ref_i = ref_mag * np.sin(ref_ph)
                meas_r = v_meas * np.cos(ph_meas)
                meas_i = v_meas * np.sin(ph_meas)
                diff = np.sqrt((ref_r-meas_r)**2 + (ref_i-meas_i)**2)
                true_mag = np.sqrt(ref_r**2 + ref_i**2)
                tve = (diff / true_mag) * 100 if true_mag > 0 else 0
                
                fe = abs(f_meas - ref_freq)
                rfe = abs(rocof_meas - ref_rocof)
                
                if t_rep > 0.1: # Settling
                    stats['max_tve'] = max(stats['max_tve'], tve)
                    stats['max_fe'] = max(stats['max_fe'], fe)
                    
                    # Step Response Logic
                    if "STEP" in test_type and t_rep >= params['t_step']:
                        if tve > 1.0:
                            stats['step_settled_time'] = t_rep

                print(f"{t_rep:<8.3f} | {v_meas:<8.2f} | {np.degrees(ph_meas):<8.3f} | {f_meas:<8.3f} | {rocof_meas:<8.3f} | {tve:<8.3f} | {fe:<8.4f} | {rfe:<8.4f}")
            
            t_current += chunk_dur

        self._verdict(test_type, params, stats)

    def _verdict(self, test_type, params, stats):
        status = "PASS"
        reasons = []
        
        if "STEP" in test_type:
            t_step = params['t_step']
            resp_time = 0.0
            if stats['step_settled_time']:
                resp_time = stats['step_settled_time'] - t_step
            # If resp_time is negative, it means it never violated limits (perfect step) or settled immediately
            resp_time = max(0.0, resp_time)
            
            if resp_time > self.RESP_TIME_LIMIT:
                status = "FAIL"
                reasons.append(f"RespTime {resp_time*1000:.1f}ms > 40ms")
            print("-" * 96)
            print(f"Step Response Time: {resp_time*1000:.2f} ms")
        else:
            if stats['max_fe'] > self.FE_LIMIT:
                status = "FAIL"
                reasons.append(f"FE > {self.FE_LIMIT}")
            if stats['max_tve'] > self.TVE_LIMIT:
                status = "FAIL"
                reasons.append(f"TVE > {self.TVE_LIMIT}%")

        if status == "FAIL":
            print(f"Result: FAIL ({', '.join(reasons)})")
        else:
            print(f"Result: PASS (Max TVE: {stats['max_tve']:.4f}%, Max FE: {stats['max_fe']:.4f} Hz)")
        print("\n")

In [ ]:
# ==========================================
# 6. MAIN RUNNER
# ==========================================
if __name__ == "__main__":
    bench = ComplianceBench()
    
    # 1. Steady State
    print("=== STEADY STATE ===")
    for vf in TEST_CONFIG['VOLTAGE_FACTORS']:
        bench.run_case(f"Voltage {vf*100}%", "STEADY", {'A': A_PEAK*vf, 'f0': F_NOMINAL})
        
    for freq in TEST_CONFIG['FREQ_POINTS']:
        bench.run_case(f"Freq {freq}Hz", "STEADY", {'A': A_PEAK, 'f0': freq})

    for h in TEST_CONFIG['HARMONIC_ORDERS']:
        bench.run_case(f"Harmonic {h}", "STEADY", {'A': A_PEAK, 'f0': F_NOMINAL, 'harmonics': h})
        
    for oob in TEST_CONFIG['OOB_FACTORS']:
        bench.run_case(f"OOB {oob*50}Hz", "STEADY", {'A': A_PEAK, 'f0': F_NOMINAL, 'oob_freq': oob*50})

    # 2. Dynamic
    print("=== DYNAMIC ===")
    bench.run_case("AM Mod 2Hz", "AM_MOD", {'A': A_PEAK, 'f0': F_NOMINAL, 'fm': 2.0}, duration=1.0)
    
    # Ramps
    r_start, r_end = TEST_CONFIG['RAMP_RANGE']
    bench.run_case(f"Ramp {r_start}-{r_end}Hz", "RAMP", {'A': A_PEAK, 'start': r_start, 'slope': 1.0}, duration=10.5)
    
    # Steps
    bench.run_case("Mag Step", "MAG_STEP", {'A': A_PEAK, 'f0': F_NOMINAL, 't_step': 0.4, 'step_size': 0.1}, duration=0.6)
    bench.run_case("Phase Step", "PHASE_STEP", {'A': A_PEAK, 'f0': F_NOMINAL, 't_step': 0.4, 'step_size': 0.1}, duration=0.6)